# ROOT Hit Conversion and Quick Analysis

This notebook loads the OpenGATE `hits.root`, builds a DataFrame similar to `convert.py`, and runs basic plots for each variable.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120
from pathlib import Path

In [ ]:
#f = uproot.open(Path("hits_offset.root"))
#tree_key = next((k for k, v in f.classnames().items() if v.endswith("TTree")), None)
#print("Tree:", tree_key)
#if tree_key:
#    tree = f[tree_key]
#    print("\n".join(tree.keys()))

In [ ]:
# Set the ROOT file path here
root_file = Path('hits_offset.root')

if not root_file.exists():
    roots = sorted(Path('.').glob('*.root'))
    raise FileNotFoundError(f'{root_file} not found. Available: {[p.name for p in roots]}')

with uproot.open(root_file) as f:
    classnames = f.classnames()
    tree_key = next((k for k, v in classnames.items() if v.endswith('TTree')), None)
    if tree_key is None:
        raise RuntimeError(f'No TTree found in {root_file}. Keys: {list(classnames.keys())}')
    tree = f[tree_key]
    arrays = tree.arrays(library='np')

def _get(name):
    return arrays[name] if name in arrays else None

def _missing(n, value=np.nan):
    return np.full(n, value)

n = len(arrays[next(iter(arrays.keys()))]) if len(arrays) > 0 else 0

def _get_vec(base):
    arr = _get(base)
    if arr is not None:
        return arr
    x = _get(f"{base}_X")
    y = _get(f"{base}_Y")
    z = _get(f"{base}_Z")
    if x is None or y is None or z is None:
        return None
    return np.stack([x, y, z], axis=1)

pos = _get_vec('Position')
if pos is not None:
    print ("got Postion!")
if pos is None:
    print ("Postion empty, getting PrePosition")
    pos = _get_vec('PrePosition')
if pos is None:
    print ("PrePostion empty, getting PostPosition")
    pos = _get_vec('PostPosition')
if pos is None:
    print ("PostPostion empty, getting PrePositionLocal")
    pos = _get_vec('PrePositionLocal')
if pos is None:
    print ("All positions empty!")
    x_mm = y_mm = z_mm = _missing(n)
else:
    x_mm, y_mm, z_mm = pos[:, 0], pos[:, 1], pos[:, 2]

edep = _get('TotalEnergyDeposit')
if edep is None:
    edep = _get('Edep')
edep_keV = edep * 1000.0 if edep is not None else _missing(n)  # assuming MeV -> keV
process = _get('ProcessDefinedStep')
if process is None:
    process = _get('TrackCreatorProcess')

# Diagnostics for process availability
if process is None:
    print('WARNING: No ProcessName or TrackCreatorProcess in ROOT file')
else:
    process_series = pd.Series(process)
    process_series = pd.Series(process)
    empty_frac = (process_series.astype(str).str.len() == 0).mean()
    print(f'Empty process fraction: {empty_frac:.1%}')

volume = _get('TrackVolumeName')
if volume is None:
    volume = _get('VolumeName')

df = pd.DataFrame(
    {
        'event': _get('EventID') if _get('EventID') is not None else _missing(n),
        'run': _get('RunID') if _get('RunID') is not None else _missing(n),
        'thread': _get('ThreadID') if _get('ThreadID') is not None else _missing(n),
        'track': _get('TrackID') if _get('TrackID') is not None else _missing(n),
        'parent': _get('ParentID') if _get('ParentID') is not None else _missing(n),
        'particle': _get('ParticleName').astype(str) if _get('ParticleName') is not None else _missing(n, ''),
        'process': process.astype(str) if process is not None else _missing(n, ''),
        'edep_keV': edep_keV,
        'x_mm': x_mm,
        'y_mm': y_mm,
        'z_mm': z_mm,
        't_ns': _get('GlobalTime') if _get('GlobalTime') is not None else _missing(n),
        'volume': volume.astype(str) if volume is not None else _missing(n, ''),
        'copyno': _get('CopyNo') if _get('CopyNo') is not None else _missing(n),
    }
)

df['interaction_type'] = np.where(
    df['process'].str.contains('phot', case=False, na=False),
    'PE',
    np.where(df['process'].str.contains('compt', case=False, na=False), 'Compton', 'Other'),
)

# Plot: energy deposited per hit by interaction type
if 'df' in globals() and 'edep_keV' in df.columns and 'interaction_type' in df.columns:
    plt.figure(figsize=(7, 4))
    for label, color in [('PE', '#1f77b4'), ('Compton', '#ff7f0e'), ('Other', '#2ca02c')]:
        vals = df.loc[df['interaction_type'] == label, 'edep_keV'].to_numpy()
        vals = vals[(~np.isnan(vals)) & (vals > 0)]
        if len(vals) == 0:
            continue
        plt.hist(vals, bins=80, alpha=0.5, label=label, color=color)
    plt.xlabel('Deposited energy per hit (keV)')
    plt.ylabel('Counts')
    plt.title('Edep per Hit by Interaction Type')
    plt.legend()
    plt.yscale('log')
    plt.tight_layout()
    plt.show()
else:
    print('df or interaction_type not available; run the data-loading cell first.')

df.head(n=80)

In [ ]:
# Event-level classification and optional Events tree
try:
    import awkward as ak
except Exception:
    ak = None

# Ensure ix/iy columns exist (derived from volume names)
if 'ix' not in df.columns or 'iy' not in df.columns:
    pixel_re = re.compile(r'pixel_(\d+)_(\d+)')
    def _extract_pixel(name):
        m = pixel_re.search(name)
        if not m:
            return np.nan, np.nan
        return float(m.group(1)), float(m.group(2))
    vol_names = df['volume'].astype(str)
    ix_iy = vol_names.apply(_extract_pixel)
    df['ix'] = [v[0] for v in ix_iy]
    df['iy'] = [v[1] for v in ix_iy]

def _classify_event(grp):
    proc = grp['process'].str.lower().fillna('')
    t_pe = grp.loc[proc.str.contains('phot'), 't_ns'].min()
    t_cs = grp.loc[proc.str.contains('compt'), 't_ns'].min()
    has_pe = np.isfinite(t_pe)
    has_cs = np.isfinite(t_cs)
    if has_pe and (not has_cs or t_pe <= t_cs):
        return 'PE'
    if has_cs:
        if has_pe and t_pe > t_cs:
            return 'CS-PE'
        return 'CS-NOPE'
    return 'Other'

df_ev = df[np.isfinite(df['event'])].copy()
if len(df_ev) == 0:
    print('No EventID found; skipping event-level classification.')
else:
    df_ev['event'] = df_ev['event'].astype(np.int64)
    event_rows = []
    for event_id, grp in df_ev.groupby('event', sort=True):
        event_rows.append({
            'EventID': event_id,
            'Class': _classify_event(grp),
            'TotalEdep_keV': np.nansum(grp['edep_keV'].to_numpy()),
            'NHits': len(grp),
        })
    events_df = pd.DataFrame(event_rows)
    display(events_df.head(n=100))
    display(events_df['Class'].value_counts())

    if ak is not None:
        # write jagged Events tree (optional)
        df_ev = df_ev.sort_values(['event', 't_ns'], kind='mergesort')
        event_ids = []
        classes = []
        total_edep = []
        hit_edep = []
        hit_t = []
        hit_x = []
        hit_y = []
        hit_z = []
        hit_ix = []
        hit_iy = []

        for event_id, grp in df_ev.groupby('event', sort=True):
            event_ids.append(event_id)
            classes.append(_classify_event(grp))
            total_edep.append(np.nansum(grp['edep_keV'].to_numpy()))
            hit_edep.append(grp['edep_keV'].to_list())
            hit_t.append(grp['t_ns'].to_list())
            hit_x.append(grp['x_mm'].to_list())
            hit_y.append(grp['y_mm'].to_list())
            hit_z.append(grp['z_mm'].to_list())
            hit_ix.append(grp['ix'].to_list())
            hit_iy.append(grp['iy'].to_list())

# NOTE: uproot cannot write jagged string arrays; process/volume strings are omitted from Events tree
        events_tree = {
            'EventID': np.array(event_ids, dtype=np.int64),
            'Class': ak.Array(classes),
            'TotalEdep_keV': np.array(total_edep, dtype=np.float64),
            'HitEdep_keV': ak.Array(hit_edep),
            'HitTime_ns': ak.Array(hit_t),
            'HitX_mm': ak.Array(hit_x),
            'HitY_mm': ak.Array(hit_y),
            'HitZ_mm': ak.Array(hit_z),
            'HitIx': ak.Array(hit_ix),
            'HitIy': ak.Array(hit_iy),
        }

        out = root_file.with_name(f'{root_file.stem}_events.root')
        with uproot.recreate(str(out)) as f:
            f['Events'] = events_tree
        print(f'Wrote {out}')
    else:
        print('awkward not installed; skipping Events tree write.')


In [ ]:
# Per-event diagnostic plots
if 'events_df' in globals() and len(events_df) > 0:
    # Class distribution
    plt.figure(figsize=(6, 4))
    events_df['Class'].value_counts().sort_index().plot(kind='bar', color='#4c72b0', alpha=0.85)
    plt.ylabel('Event count')
    plt.title('Event Class Distribution')
    plt.tight_layout()
    plt.show()

    # Total energy per event, by class
    plt.figure(figsize=(6, 4))
    for cls, g in events_df.groupby('Class'):
        vals = g['TotalEdep_keV'].to_numpy()
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        plt.hist(vals, bins=60, alpha=0.5, label=cls, density=False)
    plt.xlabel('Total deposited energy per event (keV)')
    plt.ylabel('Counts')
    plt.yscale('log')
    plt.title('Total Edep per Event by Class')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Hits per event
    plt.figure(figsize=(6, 4))
    for cls, g in events_df.groupby('Class'):
        vals = g['NHits'].to_numpy()
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        plt.hist(vals, bins=50, alpha=0.5, label=cls)
    plt.xlabel('Hits per event')
    plt.ylabel('Counts')
    plt.title('Hits per Event by Class')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Scatter: total energy vs hits per event
    plt.figure(figsize=(6, 4))
    for cls, g in events_df.groupby('Class'):
        plt.scatter(g['NHits'], g['TotalEdep_keV'], s=8, alpha=0.5, label=cls)
    plt.xlabel('Hits per event')
    plt.ylabel('Total deposited energy (keV)')
    plt.title('Edep vs Hits per Event')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('events_df not found or empty; run the event classification cell first.')


In [ ]:
df.describe(include='all').T


In [ ]:
# Heatmap of total deposited energy per pixel
pixel_re = re.compile(r'pixel_(\d+)_(\d+)')

def _extract_pixel(name):
    m = pixel_re.search(name)
    if not m:
        return np.nan, np.nan
    return float(m.group(1)), float(m.group(2))

vol_names = df['volume'].astype(str)
ix_iy = vol_names.apply(_extract_pixel)
df['ix'] = [v[0] for v in ix_iy]
df['iy'] = [v[1] for v in ix_iy]

df_pix = df[np.isfinite(df['ix']) & np.isfinite(df['iy'])].copy()
if len(df_pix) == 0:
    print('No pixel volume names found; skipping pixel heatmap.')
else:
    max_ix = int(df_pix['ix'].max())
    max_iy = int(df_pix['iy'].max())
    grid = np.zeros((max_iy + 1, max_ix + 1), dtype=float)

    for _, row in df_pix.iterrows():
        if np.isfinite(row['edep_keV']) and row['edep_keV'] > 0:
            grid[int(row['iy']), int(row['ix'])] += row['edep_keV']

    plt.figure(figsize=(5, 5))
    im = plt.imshow(grid, origin='lower', cmap='magma')
    plt.colorbar(im, label='Total deposited energy (keV)')
    plt.xlabel('pixel ix')
    plt.ylabel('pixel iy')
    plt.title('Total Edep per Pixel')
    plt.tight_layout()
    plt.show()


In [ ]:
# Basic plots for each variable
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in df.columns if c not in num_cols]

for c in num_cols:
    vals = df[c].to_numpy()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        continue
    plt.figure(figsize=(6, 4))
    plt.hist(vals, bins=100, color='#2ca02c', alpha=0.85)
    plt.xlabel(c)
    plt.ylabel('Counts')
    plt.title(f'Histogram: {c}')
    plt.tight_layout()
    plt.show()

for c in cat_cols:
    vc = df[c].value_counts().head(30)
    if len(vc) == 0:
        continue
    plt.figure(figsize=(7, 4))
    vc.sort_values().plot(kind='barh', color='#ff7f0e', alpha=0.85)
    plt.xlabel('Count')
    plt.title(f'Top categories: {c}')
    plt.tight_layout()
    plt.show()


In [ ]:
# Per-event diagnostic plots (appended at end)
if 'events_df' in globals() and len(events_df) > 0:
    # Class distribution
    plt.figure(figsize=(6, 4))
    events_df['Class'].value_counts().sort_index().plot(kind='bar', color='#4c72b0', alpha=0.85)
    plt.ylabel('Event count')
    plt.title('Event Class Distribution')
    plt.tight_layout()
    plt.show()

    # Total energy per event, by class
    plt.figure(figsize=(6, 4))
    for cls, g in events_df.groupby('Class'):
        vals = g['TotalEdep_keV'].to_numpy()
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        plt.hist(vals, bins=60, alpha=0.5, label=cls, density=False)
    plt.xlabel('Total deposited energy per event (keV)')
    plt.ylabel('Counts')
    plt.title('Total Edep per Event by Class')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Hits per event
    plt.figure(figsize=(6, 4))
    for cls, g in events_df.groupby('Class'):
        vals = g['NHits'].to_numpy()
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        plt.hist(vals, bins=50, alpha=0.5, label=cls)
    plt.xlabel('Hits per event')
    plt.ylabel('Counts')
    plt.title('Hits per Event by Class')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Scatter: total energy vs hits per event
    plt.figure(figsize=(6, 4))
    for cls, g in events_df.groupby('Class'):
        plt.scatter(g['NHits'], g['TotalEdep_keV'], s=8, alpha=0.5, label=cls)
    plt.xlabel('Hits per event')
    plt.ylabel('Total deposited energy (keV)')
    plt.title('Edep vs Hits per Event')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('events_df not found or empty; run the event classification cell first.')


In [ ]:
# Basic analysis of test019_hits.root (PhaseSpaceActor output)
phsp_file = Path('test019_hits.root')
if not phsp_file.exists():
    print('test019_hits.root not found. Run module-sim.py first.')
else:
    with uproot.open(phsp_file) as f:
        key = next((k for k, v in f.classnames().items() if v.endswith('TTree')), None)
        if key is None:
            raise RuntimeError(f'No TTree found in {phsp_file}. Keys: {list(f.keys())}')
        tree = f[key]
        arr = tree.arrays(library='np')
        tree.branches

    def _getp(name):
        return arr[name] if name in arr else None

    def _getp_vec(base):
        arr = _getp(base)
        if arr is not None:
            return arr
        x = _getp(f"{base}_X")
        y = _getp(f"{base}_Y")
        z = _getp(f"{base}_Z")
        if x is None or y is None or z is None:
            return None
        return np.stack([x, y, z], axis=1)

    posp = _getp_vec('PostPosition')
    if posp is None:
        posp = _getp_vec('Position')
    if posp is None:
        posp = _getp_vec('PrePosition')
    if posp is None:
        posp = _getp_vec('PrePositionLocal')

    dfp = pd.DataFrame({
        'event': _getp('EventID') if _getp('EventID') is not None else np.nan,
        'track': _getp('TrackID') if _getp('TrackID') is not None else np.nan,
        'parent': _getp('ParentID') if _getp('ParentID') is not None else np.nan,
        'particle': _getp('ParticleName').astype(str) if _getp('ParticleName') is not None else '',
        'process': _getp('ProcessDefinedStep').astype(str) if _getp('ProcessDefinedStep') is not None else '',
        'ekin': _getp('KineticEnergy') if _getp('KineticEnergy') is not None else np.nan,
        'time': _getp('GlobalTime') if _getp('GlobalTime') is not None else np.nan,
        'x': (posp[:, 0] if posp is not None else np.nan),
        'y': (posp[:, 1] if posp is not None else np.nan),
        'z': (posp[:, 2] if posp is not None else np.nan),
    })

    display(dfp.head(n=100))
    print('Process counts:')
    display(dfp['process'].str.lower().value_counts().head(20))

    # Simple plots
    plt.figure(figsize=(6, 4))
    dfp['ekin'].dropna().plot(kind='hist', bins=80, color='#1f77b4', alpha=0.85)
    plt.xlabel('Kinetic energy')
    plt.title('Kinetic Energy Distribution (PhaseSpace)')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    dfp['time'].dropna().plot(kind='hist', bins=80, color='#ff7f0e', alpha=0.85)
    plt.xlabel('Global time')
    plt.title('Global Time Distribution (PhaseSpace)')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(5, 5))
    plt.hist2d(dfp['x'].dropna(), dfp['y'].dropna(), bins=80, cmap='magma')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('PostPosition XY (PhaseSpace)')
    plt.colorbar(label='counts')
    plt.tight_layout()
    plt.show()


In [ ]:
# Scatter plot of hit XY with pixel grid overlay
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches

if 'df' not in globals():
    print('df not found; run the data-loading cell first.')
else:
    x = df['x_mm'].to_numpy()
    y = df['y_mm'].to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # Subsample for speed if needed
    max_points = 200000
    if len(x) > max_points:
        rng = np.random.default_rng(0)
        idx = rng.choice(len(x), size=max_points, replace=False)
        x = x[idx]
        y = y[idx]

    pix = 3.0
    foil = 0.2
    n = 8
    pitch = pix + foil
    offset = (n - 1) * pitch / 2.0

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(x, y, s=2, alpha=0.3, color='black')

    # Draw pixel rectangles
    for ix in range(n):
        for iy in range(n):
            cx = ix * pitch - offset
            cy = iy * pitch - offset
            rect = patches.Rectangle(
                (cx - pix / 2.0, cy - pix / 2.0),
                pix,
                pix,
                fill=False,
                edgecolor='tab:blue',
                linewidth=0.5,
            )
            ax.add_patch(rect)

    # Draw ESR block outline (including outer foil)
    block_xy = n * pix + (n - 1) * foil + 2 * foil
    block = patches.Rectangle(
        (-block_xy / 2.0, -block_xy / 2.0),
        block_xy,
        block_xy,
        fill=False,
        edgecolor='tab:red',
        linewidth=1.0,
    )
    ax.add_patch(block)

    ax.set_aspect('equal', 'box')
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')
    ax.set_title('Hit XY with Pixel Grid Overlay')
    plt.tight_layout()
    plt.show()
